In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import chi2, mutual_info_classif, VarianceThreshold

In [ ]:
def enhanced_filter_selection(X, y, ranking_method='mic', p=5):
    d = X.shape[1]
    k = 1
    patience = p
    step = 0
    a_star = 0
    F_star = []
    M = RandomForestClassifier()

    if ranking_method == 'chi2':
        scores, _ = chi2(X, y)
    elif ranking_method == 'mic':
        scores = mutual_info_classif(X, y)
    else:
        scores = np.var(X, axis=0)

    ranked_indices = np.argsort(scores)[::-1]

    while step < patience and k <= d:

        current_indices = ranked_indices[:k]
        X_k = X[:, current_indices]

        ak = np.mean(cross_val_score(M, X_k, y, cv=5))

        if ak > a_star:
            a_star = ak
            F_star = current_indices.tolist()
            step = 0
        else:
            step += 1

        k += 1

    return F_star, a_star